[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/ia_p26/blob/main/clase/07_optimization/notebooks/01_formulacion_y_paisaje.ipynb)

# Notebook 1: Formulación y Paisaje de Optimización

En este notebook exploramos interactivamente:
1. Cómo formular problemas de optimización
2. Paisajes 1D y 2D: mínimos, máximos, puntos silla
3. Convexidad y por qué importa

In [ ]:
# Instalar dependencias (ejecutar en Colab)
# !pip install numpy matplotlib scipy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

---
## 1. ¿Qué es optimización?

**Teoría:** [7.1 Formulación matemática](https://www.sonder.art/ia_p26/07_optimization/01_formulacion/)

Optimizar es encontrar los valores de $x$ que minimizan (o maximizan) una función $f(x)$, posiblemente con restricciones.

$$\min_{x} f(x) \quad \text{sujeto a restricciones}$$

En IA, casi todo se reduce a optimización:
- Entrenar un modelo = minimizar una función de pérdida
- Ajustar hiperparámetros = optimizar rendimiento
- Tomar decisiones = maximizar utilidad esperada

---
## 2. Ejercicio: Escribe la función objetivo

**Teoría:** [7.1 Anatomía de un problema](https://www.sonder.art/ia_p26/07_optimization/01_formulacion/#anatomía-de-un-problema-de-optimización)

**Problema:** Una tienda vende dos productos. El producto A da \$3 de ganancia y el B da \$5. Quieres maximizar la ganancia total vendiendo $x_1$ unidades de A y $x_2$ unidades de B.

**Tarea:** Completa la función objetivo en la celda siguiente.

In [ ]:
def ganancia(x1, x2):
    """Función objetivo: ganancia total."""
    return 3 * x1 + 5 * x2  # <-- CHANGE THIS if needed

# Verifica: si vendes 10 de A y 5 de B, ¿cuánto ganas?
print(f"Ganancia(10, 5) = ${ganancia(10, 5)}")
assert ganancia(10, 5) == 55, "Revisa tu función: debería ser 3*10 + 5*5 = 55"
print("¡Correcto!")

---
## 3. Visualizando paisajes 1D

**Teoría:** [7.2 Local vs global](https://www.sonder.art/ia_p26/07_optimization/02_paisaje_y_conceptos/#local-vs-global)

Una función 1D es como un terreno en una sola dimensión. Los mínimos son los valles y los máximos son las cimas.

In [ ]:
def plot_function_1d(f, x_range, title="f(x)", xlabel="x", ylabel="f(x)"):
    """Grafica una función 1D con mínimos y máximos anotados."""
    x = np.linspace(*x_range, 800)
    y = f(x)

    fig, ax = plt.subplots()
    ax.plot(x, y, linewidth=2, color="#2E86AB")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

    # Find and mark local minima (simple approach)
    from scipy.signal import argrelextrema
    min_idx = argrelextrema(y, np.less, order=20)[0]
    max_idx = argrelextrema(y, np.greater, order=20)[0]

    ax.scatter(x[min_idx], y[min_idx], color="#E94F37", s=80, zorder=5, label="Mínimos")
    ax.scatter(x[max_idx], y[max_idx], color="#27AE60", s=80, zorder=5, label="Máximos")
    ax.legend()
    plt.show()

# Función con varios mínimos locales
f1 = lambda x: (x - 2)**2 * np.sin(3 * x) + 0.5 * x
plot_function_1d(f1, (-2, 6), title=r"$f(x) = (x-2)^2 \sin(3x) + 0.5x$")

In [ ]:
# Prueba con tu propia función. Sugerencias:
# - lambda x: x**4 - 8*x**2          (¿cuántos mínimos?)
# - lambda x: np.abs(x - 2)           (¿es diferenciable en el mínimo?)
# - lambda x: x**3 - 3*x              (¿tiene mínimo global?)
f_custom = lambda x: x**2 - 4*np.cos(3*x)
plot_function_1d(f_custom, (-4, 4), title="Tu función personalizada")

### Reflexión: Paisajes 1D

- ¿Cuántos mínimos locales encontraste? ¿Alguno es el mínimo **global**?
- Si usaras descenso de gradiente empezando en $x = -3$, ¿a cuál mínimo llegarías?
- ¿El resultado depende del punto de inicio? Esto es clave para entender los límites de GD.

---
## 4. Superficies 2D y puntos silla

**Teoría:** [7.2 Puntos silla](https://www.sonder.art/ia_p26/07_optimization/02_paisaje_y_conceptos/#puntos-silla)

En 2D, podemos visualizar la función objetivo como un mapa de contornos (curvas de nivel) o como un heatmap.

Un **punto silla** es un punto donde el gradiente es cero, pero no es ni mínimo ni máximo: es mínimo en una dirección y máximo en otra.

In [ ]:
def plot_surface_2d(f, x_range, y_range, title="f(x,y)", n_levels=20):
    """Grafica contornos + heatmap de una función 2D."""
    x = np.linspace(*x_range, 200)
    y = np.linspace(*y_range, 200)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Heatmap
    im = ax1.pcolormesh(X, Y, Z, cmap="coolwarm", shading="auto")
    fig.colorbar(im, ax=ax1)
    ax1.set_title(f"{title} (heatmap)")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")

    # Contours
    cs = ax2.contour(X, Y, Z, levels=n_levels, cmap="viridis")
    ax2.clabel(cs, inline=True, fontsize=8)
    ax2.set_title(f"{title} (contornos)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y")

    fig.tight_layout()
    plt.show()

# Punto silla: f(x,y) = x^2 - y^2
saddle = lambda x, y: x**2 - y**2
plot_surface_2d(saddle, (-2, 2), (-2, 2), title=r"$f(x,y) = x^2 - y^2$ (punto silla)")

In [ ]:
# Prueba cambiar la función. Sugerencias:
# - lambda x, y: x**2*y**2 - x**2 - y**2    (¿tiene puntos silla?)
# - lambda x, y: np.sin(x) + np.sin(y)        (¿cuántos mínimos locales ves?)
# - lambda x, y: (x**2 + y - 11)**2 + (x + y**2 - 7)**2  (Himmelblau: 4 mínimos)
f_custom_2d = lambda x, y: np.sin(x) * np.cos(y)
plot_surface_2d(f_custom_2d, (-4, 4), (-4, 4), title="Tu función 2D")

### Reflexión: Superficies 2D

- ¿La función que probaste tiene punto silla, bowl (mínimo claro), o múltiples mínimos?
- ¿Cómo distingues un punto silla de un mínimo en los contornos? (Pista: en un mínimo los contornos son curvas cerradas anidadas)
- ¿Por qué los puntos silla son más problemáticos que los mínimos locales en alta dimensión?

---
## 5. Convexidad

**Teoría:** [7.2 Convexidad](https://www.sonder.art/ia_p26/07_optimization/02_paisaje_y_conceptos/#convexidad)

Una función es **convexa** si la cuerda entre dos puntos de la curva siempre queda **por arriba** de la curva:

$$f(\lambda x + (1-\lambda)y) \leq \lambda f(x) + (1-\lambda) f(y) \quad \forall \lambda \in [0,1]$$

**Importancia clave:** En funciones convexas, todo mínimo local es global.

In [ ]:
def check_convexity_visual(f, x_range, title="", n_samples=5):
    """Test visual de convexidad: dibuja cuerdas y verifica si quedan arriba de la curva."""
    x = np.linspace(*x_range, 500)
    y = f(x)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, y, linewidth=2, color="#2E86AB", label="f(x)")

    np.random.seed(42)
    is_convex = True
    for _ in range(n_samples):
        x1, x2 = sorted(np.random.uniform(*x_range, 2))
        y1, y2 = f(x1), f(x2)

        # Check midpoint
        lam = 0.5
        x_mid = lam * x1 + (1 - lam) * x2
        chord_val = lam * y1 + (1 - lam) * y2
        func_val = f(x_mid)

        color = "#27AE60" if func_val <= chord_val + 1e-10 else "#E94F37"
        if func_val > chord_val + 1e-10:
            is_convex = False

        ax.plot([x1, x2], [y1, y2], "--", color=color, alpha=0.6, linewidth=1)

    verdict = "CONVEXA" if is_convex else "NO CONVEXA"
    ax.set_title(f"{title} — {verdict} (verde = cuerda arriba, rojo = violación)")
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.legend()
    plt.show()

# Convexa
check_convexity_visual(lambda x: x**2, (-3, 3), title=r"$f(x) = x^2$")

In [ ]:
# No convexa
check_convexity_visual(lambda x: np.sin(x), (-4, 4), title=r"$f(x) = \sin(x)$")

In [ ]:
# Convex vs non-convex side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

x = np.linspace(-3, 3, 400)

# Convex: x^2
ax1.plot(x, x**2, linewidth=2, color="#2E86AB")
ax1.fill_between(x, x**2, alpha=0.1, color="#2E86AB")
# Chord
x1, x2 = -2, 1.5
ax1.plot([x1, x2], [x1**2, x2**2], "--", color="#E94F37", linewidth=1.5, label="Cuerda")
ax1.set_title(r"Convexa: $f(x) = x^2$")
ax1.legend()

# Non-convex: x^4 - 4x^2
y_nc = x**4 - 4*x**2 + x
ax2.plot(x, y_nc, linewidth=2, color="#8E44AD")
x1, x2 = -1.8, 1.5
y1_nc = x1**4 - 4*x1**2 + x1
y2_nc = x2**4 - 4*x2**2 + x2
ax2.plot([x1, x2], [y1_nc, y2_nc], "--", color="#E94F37", linewidth=1.5, label="Cuerda")
ax2.set_title(r"No convexa: $f(x) = x^4 - 4x^2 + x$")
ax2.legend()

fig.suptitle("Convexidad: la cuerda siempre arriba de la curva", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

---
## 6. Ejercicio: Clasifica funciones

Para cada función, grafícala y determina si es convexa o no convexa.

In [ ]:
# Función 1: f(x) = |x|
check_convexity_visual(lambda x: np.abs(x), (-3, 3), title=r"$f(x) = |x|$")

In [ ]:
# Función 2: f(x) = e^x
check_convexity_visual(lambda x: np.exp(x), (-3, 3), title=r"$f(x) = e^x$")

In [ ]:
# Función 3: f(x) = cos(x)
check_convexity_visual(lambda x: np.cos(x), (-4, 4), title=r"$f(x) = \cos(x)$")

In [ ]:
# Función 4: tu propia función
f_test = lambda x: x**2 + 2*np.abs(x)  # <-- CHANGE THIS
check_convexity_visual(f_test, (-3, 3), title="Tu función")

---
## 7. Formulando problemas reales

**Teoría:** [7.1 Receta para formular](https://www.sonder.art/ia_p26/07_optimization/01_formulacion/#receta-para-formular)

**Receta:**
1. ¿Qué controlas? → Variables de decisión $x$
2. ¿Qué quieres optimizar? → Función objetivo $f(x)$
3. ¿Qué limitaciones hay? → Restricciones

---
## 8. Ejercicio: Problema de producción

**Teoría:** [7.1 Ejemplo 1: Producción LP](https://www.sonder.art/ia_p26/07_optimization/01_formulacion/#ejemplos)

Fábrica con 2 productos. Producto 1 da \$5 de ganancia, producto 2 da \$4.
- Recurso A: producto 1 usa 6 unidades, producto 2 usa 4. Disponible: 24.
- Recurso B: producto 1 usa 1 unidad, producto 2 usa 2. Disponible: 6.
- No se puede producir cantidades negativas.

**Tarea:** Escribe las matrices $c$, $A$, $b$ como arrays de numpy.

---
## 8. Ejercicio: Problema de producción

Fábrica con 2 productos. Producto 1 da \\$5 de ganancia, producto 2 da \\$4.
- Recurso A: producto 1 usa 6 unidades, producto 2 usa 4. Disponible: 24.
- Recurso B: producto 1 usa 1 unidad, producto 2 usa 2. Disponible: 6.
- No se puede producir cantidades negativas.

**Tarea:** Escribe las matrices $c$, $A$, $b$ como arrays de numpy.

In [ ]:
# Completa las matrices del problema LP: max c^T x  s.t. Ax <= b, x >= 0
c = np.array([5, 4])                      # <-- coeficientes del objetivo
A = np.array([[6, 4], [1, 2]])             # <-- CHANGE THIS: matriz de restricciones
b = np.array([24, 6])                      # <-- CHANGE THIS: lado derecho

# Verificación
assert c.shape == (2,), f"c debe ser (2,), tienes {c.shape}"
assert A.shape == (2, 2), f"A debe ser (2,2), tienes {A.shape}"
assert b.shape == (2,), f"b debe ser (2,), tienes {b.shape}"

# Test: ¿es factible x = (2, 1)?
x_test = np.array([2, 1])
print(f"Ax = {A @ x_test}, b = {b}")
print(f"¿Factible? {np.all(A @ x_test <= b) and np.all(x_test >= 0)}")
print(f"Ganancia: {c @ x_test}")

# Test: ¿es factible x = (3, 1.5)?
x_opt = np.array([3, 1.5])
print(f"\nAx = {A @ x_opt}, b = {b}")
print(f"¿Factible? {np.all(A @ x_opt <= b) and np.all(x_opt >= 0)}")
print(f"Ganancia: {c @ x_opt}")